In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/piyushchaudhari007/sentiment-data/bitcoin_sentiments_21_24.csv
/kaggle/input/datasets/piyushchaudhari007/sentiment-data/crypto-news-headlines.csv


In [2]:
df1= pd.read_csv('/kaggle/input/datasets/piyushchaudhari007/sentiment-data/bitcoin_sentiments_21_24.csv')
df2= pd.read_csv('/kaggle/input/datasets/piyushchaudhari007/sentiment-data/crypto-news-headlines.csv')

In [3]:
import pandas as pd

# 1. Process df1: Map continuous sentiment scores to discrete labels
# Positive > 0 (1), Neutral == 0 (0), Negative < 0 (-1)
def label_df1(score):
    if score > 0:
        return 1
    elif score < 0:
        return -1
    else:
        return 0

df1['label'] = df1['Accurate Sentiments'].apply(label_df1)

# 2. Process df2: Re-map existing integer labels to match our scheme
# df2 current: Positive=2, Neutral=0, Negative=1
# We want: Positive=1, Neutral=0, Negative=-1
mapping_df2 = {2: 1, 0: 0, 1: -1}
df2['label'] = df2['label'].map(mapping_df2)

# 3. Standardize column names before merging
# Renaming 'Short Description' in df1 and 'text' in df2 to a common name like 'content'
df1_clean = df1[['Short Description', 'label']].rename(columns={'Short Description': 'text'})
df2_clean = df2[['text', 'label']]

# 4. Concatenate the two datasets
merged_df = pd.concat([df1_clean, df2_clean], ignore_index=True)

# Display result
print(merged_df.head())
print(merged_df['label'].value_counts())

                                                text  label
0  Bitcoin price is consolidating near the USD 62...      1
1  Congress could finally approve or reject the m...      0
2  Bitcoin increasingly becoming a political inst...      0
3  There is still potential for the price of bitc...      1
4  'Several companies' are looking to Latin Ameri...      0
label
 0    4620
 1    4124
-1    3069
Name: count, dtype: int64


In [4]:
# 1. Identify the size of the smallest class
min_count = merged_df['label'].value_counts().min()  # This will be 3069

# 2. Balance the dataset
# We group by label and take a random sample of 'min_count' from each group
balanced_df = (merged_df.groupby('label', group_keys=False)
               .apply(lambda x: x.sample(n=min_count, random_state=42)))

# 3. Shuffle the final dataframe (optional but recommended for training)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the results
print("Balanced Label Counts:")
print(balanced_df['label'].value_counts())

Balanced Label Counts:
label
-1    3069
 0    3069
 1    3069
Name: count, dtype: int64


/tmp/ipykernel_55/2131608942.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=42)))


In [5]:
!pip install torch

In [21]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    pipeline,
    EarlyStoppingCallback # Added for anti-overfitting
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# 1. Environment setup
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Prep (Map: 0=Neutral, 1=Positive, 2=Negative)
balanced_df['label'] = balanced_df['label'].replace(-1, 2)
dataset = Dataset.from_pandas(balanced_df[['text', 'label']])
dataset = dataset.train_test_split(test_size=0.15)

# 3. Tokenization
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_ds = dataset.map(tokenize_fn, batched=True)

# 4. Anti-Overfitting Training Configuration
training_args = TrainingArguments(
    output_dir="./crypto_model_multi_gpu",
    eval_strategy="epoch",        
    save_strategy="epoch",        
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    fp16=True,                    
    dataloader_num_workers=2,
    num_train_epochs=10,           # Set higher, EarlyStopping will cut it short
    weight_decay=0.01,
    load_best_model_at_end=True,   # Reverts to the best version before overfitting
    metric_for_best_model="eval_loss",
    report_to="none"
)

# 5. Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

# 6. Trainer with Early Stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    # Stops if model doesn't improve for 2 straight evaluations
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

# 7. Execute Training
print("Starting Training with Early Stopping...")
trainer.train()

# --- SELF-LEARNING LOGIC ---
# This function identifies new data the model is VERY sure about
def get_pseudo_labels(model_path, unlabeled_texts, threshold=0.97):
    temp_pipe = pipeline("sentiment-analysis", model=model_path, device=0)
    new_samples = []
    results = temp_pipe(unlabeled_texts)
    
    for text, res in zip(unlabeled_texts, results):
        if res['score'] > threshold:
            # Map string back to integer for future retraining
            label_map = {"neutral": 0, "Bullish": 1, "Bearish": 2}
            new_samples.append({"text": text, "label": label_map[res['label'].lower()]})
    
    return pd.DataFrame(new_samples)

# Save the Best Model
trainer.save_model("./final_crypto_model_v1")
tokenizer.save_pretrained("./final_crypto_model_v1")

# 8. Formal Evaluation
print("\nGenerating predictions on test set...")
predictions = trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
acc = accuracy_score(labels, preds)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [f"{acc:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}"]
})
print("\n--- Final Model Performance ---")
print(metrics_df.to_string(index=False))

# 9. Sarcasm & Logic Test
pipe = pipeline("sentiment-analysis", model="./final_crypto_model_v1", device=0)
model_labels = pipe.model.config.id2label

test_headlines = [
    "Bitcoin hits $100k, we are all going to be rich!", 
    "Oh great, another 'stablecoin' just de-pegged to zero. Fantastic.", 
    "The market is sideways, nothing is happening today.", 
    "I love losing my life savings on monkey JPEGs.", 
    "Regulatory clarity is finally coming to the sector."
]

print("\n--- Live Sarcasm & Logic Test ---")
for text in test_headlines:
    result = pipe(text)[0]
    raw_label = result['label']
    confidence = result['score']
    
    # Resolve Label mapping
    if raw_label in model_labels.values():
        final_sentiment = raw_label
    else:
        label_id = int(raw_label.split('_')[-1])
        final_sentiment = model_labels[label_id]

    display_sentiment = final_sentiment.capitalize()
    if confidence < 0.45:
        display_sentiment = f"Neutral (Low Confidence: {confidence:.2f})"

    print(f"Text: {text}\nResult: {display_sentiment} (Confidence: {confidence:.4f})\n")

Map:   0%|          | 0/7825 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Training with Early Stopping...


...


Generating predictions on test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Final Model Performance ---
   Metric  Score
 Accuracy 0.8184
Precision 0.8281
   Recall 0.8184
 F1-Score 0.8181


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Live Sarcasm & Logic Test ---
Text: Bitcoin hits $100k, we are all going to be rich!
Result: Positive (Confidence: 0.8177)

Text: Oh great, another 'stablecoin' just de-pegged to zero. Fantastic.
Result: Neutral (Confidence: 0.4535)

Text: The market is sideways, nothing is happening today.
Result: Positive (Confidence: 0.8087)

Text: I love losing my life savings on monkey JPEGs.
Result: Neutral (Confidence: 0.5120)

Text: Regulatory clarity is finally coming to the sector.
Result: Positive (Confidence: 0.9769)



In [22]:
import os

# 1. Define the output path
model_export_path = "./final_crypto_sarcasm_model_v1"

# 2. Create the directory if it doesn't exist
if not os.path.exists(model_export_path):
    os.makedirs(model_export_path)

# 3. Save the Model Weights (The "Brain")
# This saves the fine-tuned parameters from your training session
trainer.save_model(model_export_path)

# 4. Save the Tokenizer (The "Translator")
# Without this, the model won't understand your text input later
tokenizer.save_pretrained(model_export_path)

# 5. Save the Config (The "Instructions")
# This ensures Negative, Neutral, and Positive labels stay in the right order
model.config.save_pretrained(model_export_path)

print(f"✅ Model successfully saved to: {model_export_path}")
print("Files created: config.json, model.safetensors, tokenizer.json, vocab.txt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model successfully saved to: ./final_crypto_sarcasm_model_v1
Files created: config.json, model.safetensors, tokenizer.json, vocab.txt


In [23]:
# Use the pipeline with your improved sarcasm model
pipe = pipeline("sentiment-analysis", model="./final_crypto_sarcasm_model_v1", device=0)

# Get the mapping from the model
id2label = pipe.model.config.id2label

test_headlines = [
    "Bitcoin hits $100k, we are all going to be rich!", 
    "Oh great, another 'stablecoin' just de-pegged to zero. Fantastic.", 
    "The market is sideways, nothing is happening today.", 
    "I love losing my life savings on monkey JPEGs.", 
    "Regulatory clarity is finally coming to the sector."
]

print("\n--- Final Fixed Sarcasm & Logic Test ---")

for text in test_headlines:
    result = pipe(text)[0]
    raw_label = result['label']  # This is either 'positive' or 'LABEL_1'
    score = result['score']
    
    # 1. Handle string labels directly (FinBERT style)
    if raw_label.lower() in ['Bullish', 'Bearish', 'neutral']:
        final_sentiment = raw_label.capitalize()
    
    # 2. Handle generic labels (BERT default style)
    elif "LABEL_" in raw_label:
        label_id = int(raw_label.split('_')[-1])
        final_sentiment = id2label[label_id].capitalize()
    
    # 3. Fallback
    else:
        final_sentiment = raw_label.capitalize()

    # Apply a logic check for Sarcasm: 
    # If the model is unsure (Confidence < 0.45), we label as Neutral
    if score < 0.45:
        final_sentiment = "Neutral (Low Confidence)"

    print(f"Text: {text}")
    print(f"Result: {final_sentiment} (Confidence: {score:.4f})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Final Fixed Sarcasm & Logic Test ---
Text: Bitcoin hits $100k, we are all going to be rich!
Result: Positive (Confidence: 0.8177)

Text: Oh great, another 'stablecoin' just de-pegged to zero. Fantastic.
Result: Neutral (Confidence: 0.4535)

Text: The market is sideways, nothing is happening today.
Result: Positive (Confidence: 0.8087)

Text: I love losing my life savings on monkey JPEGs.
Result: Neutral (Confidence: 0.5120)

Text: Regulatory clarity is finally coming to the sector.
Result: Positive (Confidence: 0.9769)



In [24]:
import os

# 1. Define the output path
model_export_path = "./final_crypto_sarcasm_model_v1"

# 2. Create the directory if it doesn't exist
if not os.path.exists(model_export_path):
    os.makedirs(model_export_path)

# 3. Save the Model Weights (The "Brain")
# This saves the fine-tuned parameters from your training session
trainer.save_model(model_export_path)

# 4. Save the Tokenizer (The "Translator")
# Without this, the model won't understand your text input later
tokenizer.save_pretrained(model_export_path)

# 5. Save the Config (The "Instructions")
# This ensures Negative, Neutral, and Positive labels stay in the right order
model.config.save_pretrained(model_export_path)

print(f"✅ Model successfully saved to: {model_export_path}")
print("Files created: config.json, model.safetensors, tokenizer.json, vocab.txt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model successfully saved to: ./final_crypto_sarcasm_model_v1
Files created: config.json, model.safetensors, tokenizer.json, vocab.txt


In [25]:
import shutil
import os
from IPython.display import FileLink

# 1. Define the folder you want to zip
model_folder = './final_crypto_sarcasm_model_v1'
zip_name = 'crypto_sarcasm_model_complete'

# 2. Create the zip file
# This will create 'crypto_sarcasm_model_complete.zip' in your current directory
shutil.make_archive(zip_name, 'zip', model_folder)

# 3. Generate a clickable download link (Works in Kaggle/Colab)
if os.path.exists(f"{zip_name}.zip"):
    print(f"✅ Success! Your model has been zipped.")
    print("Click the link below to download:")
    display(FileLink(f"{zip_name}.zip"))
else:
    print("❌ Error: Zip file was not created.")

✅ Success! Your model has been zipped.
Click the link below to download:


/kaggle/working/crypto_sarcasm_model_complete.zip

In [7]:
# import torch
# import numpy as np
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# from transformers import pipeline

# # 1. Run Predictions on the Test Set
# print("Generating predictions on test set...")
# predictions = trainer.predict(tokenized_ds["test"])
# preds = np.argmax(predictions.predictions, axis=1)
# labels = predictions.label_ids

# # 2. Calculate Metrics
# precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
# acc = accuracy_score(labels, preds)

# # 3. Display Results Table
# metrics_df = pd.DataFrame({
#     "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
#     "Score": [f"{acc:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}"]
# })

# print("\n--- Final Model Performance ---")
# print(metrics_df.to_string(index=False))

# # 4. The "Sarcasm & Stress Test" (FIXED LOGIC)
# pipe = pipeline("sentiment-analysis", model="./final_crypto_model_v1", device=0)

# # Extract the ACTUAL labels from the model config to avoid index errors
# # Usually: {0: 'neutral', 1: 'positive', 2: 'negative'} but let's be sure
# model_labels = pipe.model.config.id2label

# test_headlines = [
#     "Bitcoin hits $100k, we are all going to be rich!", 
#     "Oh great, another 'stablecoin' just de-pegged to zero. Fantastic.", 
#     "The market is sideways, nothing is happening today.", 
#     "I love losing my life savings on monkey JPEGs.", 
#     "Regulatory clarity is finally coming to the sector."
# ]

# print("\n--- Live Sarcasm & Logic Test ---")
# for text in test_headlines:
#     result = pipe(text)[0]
#     raw_label = result['label']
#     confidence = result['score']
    
#     # Logic to handle both 'LABEL_X' and string labels
#     if raw_label in model_labels.values():
#         final_sentiment = raw_label
#     else:
#         # Extract the integer from 'LABEL_0' etc.
#         label_id = int(raw_label.split('_')[-1])
#         final_sentiment = model_labels[label_id]

#     # OPTIONAL: Handle low confidence (below 45%) as Neutral
#     display_sentiment = final_sentiment.capitalize()
#     if confidence < 0.45:
#         display_sentiment = f"Neutral (Ambiguous - Confidence too low: {confidence:.2f})"

#     print(f"Text: {text}")
#     print(f"Result: {display_sentiment} (Confidence: {confidence:.4f})\n")